<a href="https://colab.research.google.com/github/Lakshay-Juneja/dva_project/blob/data/ipl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

new start from here

In [ ]:
df = pd.read_excel("/IPL.xlsx")
df.head()

,Unnamed: 0,match_id,date,match_type,event_name,innings,batting_team,bowling_team,over,ball,...,team_runs,team_balls,team_wicket,new_batter,batter_runs,batter_balls,bowler_wicket,batting_partners,next_batter,striker_out
0,131970,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,...,1,1,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
1,131971,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,...,1,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
2,131972,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
3,131973,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,3,0,NaN,0,2,0,"('BB McCullum', 'SC Ganguly')",NaN,False
4,131974,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,...,2,4,0,NaN,0,3,0,"('BB McCullum', 'SC Ganguly')",NaN,False


In [ ]:
df.columns = df.columns.str.lower().str.strip()

df['runs_total'] = pd.to_numeric(df['runs_total'], errors='coerce')
df['runs_batter'] = pd.to_numeric(df['runs_batter'], errors='coerce')

df['is_wicket'] = df['wicket_kind'].notna().astype(int)

df.head()

,unnamed: 0,match_id,date,match_type,event_name,innings,batting_team,bowling_team,over,ball,...,team_balls,team_wicket,new_batter,batter_runs,batter_balls,bowler_wicket,batting_partners,next_batter,striker_out,is_wicket
0,131970,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,...,1,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False,0
1,131971,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,...,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False,0
2,131972,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False,0
3,131973,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,3,0,NaN,0,2,0,"('BB McCullum', 'SC Ganguly')",NaN,False,0
4,131974,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,...,4,0,NaN,0,3,0,"('BB McCullum', 'SC Ganguly')",NaN,False,0


In [ ]:
df.columns = df.columns.str.lower().str.strip()

# Convert numeric columns
num_cols = ['runs_batter','runs_extras','runs_total','season','year']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Wicket flag
df['is_wicket'] = df['wicket_kind'].notna().astype(int)

overview_matches = pd.DataFrame({
    "Metric": ["Total Matches"],
    "Value": [df['match_id'].nunique()]
})

overview_matches


,Metric,Value
0,Total Matches,1169


In [ ]:
overview_matches = pd.DataFrame({
    "Metric": ["Total Matches"],
    "Value": [df['match_id'].nunique()]
})
overview_matches

,Metric,Value
0,Total Matches,1169


In [ ]:
overview_runs = pd.DataFrame({
    "Metric": ["Total Runs"],
    "Value": [df['runs_total'].sum()]
})
overview_runs

,Metric,Value
0,Total Runs,374283


In [ ]:
fours = (df['runs_batter'] == 4).sum()
sixes = (df['runs_batter'] == 6).sum()

overview_boundaries = pd.DataFrame({
    "Metric": ["Fours","Sixes"],
    "Value": [fours, sixes]
})
overview_boundaries

,Metric,Value
0,Fours,32113
1,Sixes,14353


In [ ]:
overview_teams = pd.DataFrame({
    "Metric":["Total Teams"],
    "Value":[df['batting_team'].nunique()]
})
overview_teams

,Metric,Value
0,Total Teams,19


In [ ]:
def era_map(season):
    if 2008 <= season <= 2012:
        return "2008-2012"
    elif 2013 <= season <= 2017:
        return "2013-2017"
    elif 2018 <= season <= 2022:
        return "2018-2022"
    else:
        return "2023-2025"

df['era'] = df['season'].apply(era_map)
print(era_map(2010))

2008-2012


In [ ]:
match_runs = df.groupby(['era','match_id'])['runs_total'].sum().reset_index()

era_avg_score = match_runs.groupby('era')['runs_total'].mean().reset_index()
era_avg_score.rename(columns={'runs_total':'avg_match_score'}, inplace=True)
match_runs
era_avg_score

,era,avg_match_score
0,2008-2012,293.921569
1,2013-2017,310.617834
2,2018-2022,324.279528
3,2023-2025,338.594458


In [ ]:
balls = df.groupby(['era','match_id']).size().reset_index(name='balls')
runs = df.groupby(['era','match_id'])['runs_total'].sum().reset_index()

rr = pd.merge(runs, balls, on=['era','match_id'])
rr['run_rate'] = (rr['runs_total']/rr['balls'])*6

era_runrate = rr.groupby('era')['run_rate'].mean().reset_index()
balls
runs
rr

,era,match_id,runs_total,balls,run_rate
0,2008-2012,392181,311,244,7.647541
1,2008-2012,392182,191,221,5.185520
2,2008-2012,392183,162,108,9.000000
3,2008-2012,392184,205,199,6.180905
4,2008-2012,392185,266,220,7.254545
...,...,...,...,...,...
1164,2023-2025,1473508,207,147,8.448980
1165,2023-2025,1473509,436,248,10.548387
1166,2023-2025,1473510,410,244,10.081967
1167,2023-2025,1473511,374,252,8.904762


In [ ]:
df['is_six'] = (df['runs_batter'] == 6).astype(int)

era_sixes = df.groupby('era')['is_six'].sum().reset_index()
era_sixes

,era,is_six
0,2008-2012,1880
1,2013-2017,3433
2,2018-2022,3407
3,2023-2025,5633


In [ ]:
era_wickets = df.groupby('era')['is_wicket'].sum().reset_index()
era_wickets

,era,is_wicket
0,2008-2012,2369
1,2013-2017,3654
2,2018-2022,3036
3,2023-2025,4764


In [ ]:
team_season_runs = df.groupby(
    ['season','batting_team']
)['runs_total'].sum().reset_index()
team_season_runs

,season,batting_team,runs_total
0,2009.0,Chennai Super Kings,2231
1,2009.0,Deccan Chargers,2408
2,2009.0,Delhi Daredevils,2131
3,2009.0,Kings XI Punjab,1928
4,2009.0,Kolkata Knight Riders,1772
...,...,...,...
127,2025.0,Mumbai Indians,2912
128,2025.0,Punjab Kings,3262
129,2025.0,Rajasthan Royals,2614
130,2025.0,Royal Challengers Bengaluru,2653


In [ ]:
team_wickets = df.groupby(
    ['season','bowling_team']
)['is_wicket'].sum().reset_index()
team_wickets

,season,bowling_team,is_wicket
0,2009.0,Chennai Super Kings,91
1,2009.0,Deccan Chargers,110
2,2009.0,Delhi Daredevils,106
3,2009.0,Kings XI Punjab,76
4,2009.0,Kolkata Knight Riders,59
...,...,...,...
127,2025.0,Mumbai Indians,116
128,2025.0,Punjab Kings,97
129,2025.0,Rajasthan Royals,68
130,2025.0,Royal Challengers Bengaluru,94


In [ ]:
team_match_runs = df.groupby(
    ['batting_team','match_id']
)['runs_total'].sum().reset_index()

team_avg_score = team_match_runs.groupby(
    'batting_team'
)['runs_total'].mean().reset_index()
team_match_runs
team_avg_score

,batting_team,runs_total
0,Chennai Super Kings,163.625498
1,Deccan Chargers,152.840000
2,Delhi Capitals,165.714286
3,Delhi Daredevils,150.906832
4,Gujarat Lions,162.066667
5,Gujarat Titans,177.483333
6,Kings XI Punjab,158.231579
7,Kochi Tuskers Kerala,135.785714
8,Kolkata Knight Riders,156.564394
9,Lucknow Super Giants,176.586207


In [ ]:
team_boundaries = df[df['runs_batter'].isin([4,6])] \
    .groupby('batting_team')['runs_batter'] \
    .count() \
    .reset_index(name='boundary_count')
team_boundaries

,batting_team,boundary_count
0,Chennai Super Kings,5006
1,Deccan Chargers,1357
2,Delhi Capitals,2233
3,Delhi Daredevils,2957
4,Gujarat Lions,615
5,Gujarat Titans,1342
6,Kings XI Punjab,3706
7,Kochi Tuskers Kerala,223
8,Kolkata Knight Riders,5230
9,Lucknow Super Giants,1279


In [ ]:
top_batters = df.groupby('batter')['runs_batter'] \
    .sum() \
    .sort_values(ascending=False) \
    .head(10) \
    .reset_index()
top_batters

,batter,runs_batter
0,V Kohli,8671
1,RG Sharma,7048
2,S Dhawan,6769
3,DA Warner,6567
4,SK Raina,5536
5,MS Dhoni,5439
6,KL Rahul,5235
7,AB de Villiers,5181
8,AM Rahane,5032
9,CH Gayle,4997


In [ ]:
top_bowlers = df.groupby('bowler')['is_wicket'] \
    .sum() \
    .sort_values(ascending=False) \
    .head(10) \
    .reset_index()
top_bowlers

,bowler,is_wicket
0,YS Chahal,229
1,B Kumar,213
2,SP Narine,212
3,DJ Bravo,207
4,R Ashwin,205
5,JJ Bumrah,203
6,PP Chawla,201
7,SL Malinga,188
8,A Mishra,183
9,RA Jadeja,179


In [4]:
import pandas as pd
df = pd.read_excel("/IPL.xlsx")
df.columns = df.columns.str.lower().str.strip()
# Keep only necessary columns for dashboard
df = df[[
    'match_id',
    'season',
    'batting_team',
    'bowling_team',
    'runs_batter',
    'runs_total',
    'wicket_kind'
]]

# Convert numeric
df['runs_batter'] = pd.to_numeric(df['runs_batter'], errors='coerce')
df['runs_total'] = pd.to_numeric(df['runs_total'], errors='coerce')

# Wicket flag
df['is_wicket'] = df['wicket_kind'].notna().astype(int)
df.head()

,match_id,season,batting_team,bowling_team,runs_batter,runs_total,wicket_kind,is_wicket
0,335982,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,NaN,0
1,335982,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,0,0,NaN,0
2,335982,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,NaN,0
3,335982,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,0,0,NaN,0
4,335982,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,0,0,NaN,0


In [38]:
def year_summary(year):

    data = df[df['season'] == year].copy()

    if data.empty:
        print("No data available for this year.")
        return

    print(f"\nSUMMARY FOR {year}\n")

    print("Total Matches:")
    print(data['match_id'].nunique())

    print("\nTotal Sixes:")
    print((data['runs_batter'] == 6).sum())

    print("\nTotal Fours:")
    print((data['runs_batter'] == 4).sum())

    print("\nTeam Runs:")
    print(data.groupby('batting_team')['runs_total'].sum())
year_summary(2012)


SUMMARY FOR 2012

Total Matches:
74

Total Sixes:
733

Total Fours:
1911

Team Runs:
batting_team
Chennai Super Kings            2831
Deccan Chargers                2312
Delhi Daredevils               2645
Kings XI Punjab                2390
Kolkata Knight Riders          2504
Mumbai Indians                 2462
Pune Warriors                  2321
Rajasthan Royals               2516
Royal Challengers Bangalore    2472
Name: runs_total, dtype: int64


In [39]:
import pandas as pd

# Ensure numeric columns
df['season'] = pd.to_numeric(df['season'], errors='coerce')
df['runs_total'] = pd.to_numeric(df['runs_total'], errors='coerce')
df['runs_batter'] = pd.to_numeric(df['runs_batter'], errors='coerce')

# ------------------------------
# 1️⃣ Total Matches Per Year
# ------------------------------
matches_per_year = df.groupby('season')['match_id'] \
    .nunique() \
    .reset_index(name='Total Matches')

# ------------------------------
# 2️⃣ Average Match Score Per Year
# ------------------------------
match_runs = df.groupby(['season','match_id'])['runs_total'] \
    .sum() \
    .reset_index()

avg_score = match_runs.groupby('season')['runs_total'] \
    .mean() \
    .reset_index(name='Avg Match Score')

# ------------------------------
# 3️⃣ Run Rate Per Year
# ------------------------------
balls = df.groupby(['season','match_id']).size().reset_index(name='Balls')
runs = df.groupby(['season','match_id'])['runs_total'].sum().reset_index()

rr = pd.merge(runs, balls, on=['season','match_id'])
rr['Run Rate'] = (rr['runs_total'] / rr['Balls']) * 6

avg_runrate = rr.groupby('season')['Run Rate'] \
    .mean() \
    .reset_index()

# ------------------------------
# 4️⃣ Total Sixes Per Year
# ------------------------------
sixes = df[df['runs_batter'] == 6] \
    .groupby('season') \
    .size() \
    .reset_index(name='Total Sixes')

# ------------------------------
# 5️⃣ Total Fours Per Year
# ------------------------------
fours = df[df['runs_batter'] == 4] \
    .groupby('season') \
    .size() \
    .reset_index(name='Total Fours')

# ------------------------------
# 6️⃣ Merge Everything Into One Table
# ------------------------------
year_summary_table = matches_per_year \
    .merge(avg_score, on='season') \
    .merge(avg_runrate, on='season') \
    .merge(sixes, on='season') \
    .merge(fours, on='season')

# ------------------------------
# 7️⃣ Sort Year Wise
# ------------------------------
year_summary_table = year_summary_table \
    .sort_values('season') \
    .reset_index(drop=True)

year_summary_table

,season,Total Matches,Avg Match Score,Run Rate,Total Sixes,Total Fours
0,2009.0,57,286.894737,7.227063,508,1321
1,2011.0,73,289.780822,7.411735,639,1916
2,2012.0,74,303.418919,7.573468,733,1911
3,2013.0,76,297.394737,7.470888,681,2054
4,2014.0,60,315.516667,7.922345,715,1563
5,2015.0,59,311.067797,8.134891,692,1611
6,2016.0,60,314.366667,8.000792,639,1633
7,2017.0,59,318.406780,8.088623,706,1612
8,2018.0,60,331.683333,8.362224,872,1652
9,2019.0,60,323.900000,8.177726,786,1655


In [40]:
import pandas as pd

# Ensure numeric columns
df['season'] = pd.to_numeric(df['season'], errors='coerce')
df['runs_total'] = pd.to_numeric(df['runs_total'], errors='coerce')
df['runs_batter'] = pd.to_numeric(df['runs_batter'], errors='coerce')

# Create wicket flag if not already created
df['is_wicket'] = df['wicket_kind'].notna().astype(int)

# ---------------------------------------------------
# 1️⃣ Matches Played Per Team Per Season
# ---------------------------------------------------
matches = df.groupby(['season','batting_team'])['match_id'] \
    .nunique() \
    .reset_index(name='Matches Played')

# ---------------------------------------------------
# 2️⃣ Total Runs Per Team Per Season
# ---------------------------------------------------
team_runs = df.groupby(['season','batting_team'])['runs_total'] \
    .sum() \
    .reset_index(name='Total Runs')

# ---------------------------------------------------
# 3️⃣ Avg Match Score (Correct Method)
# ---------------------------------------------------
match_runs = df.groupby(['season','batting_team','match_id'])['runs_total'] \
    .sum() \
    .reset_index()

avg_score = match_runs.groupby(['season','batting_team'])['runs_total'] \
    .mean() \
    .reset_index(name='Avg Match Score')

# ---------------------------------------------------
# 4️⃣ Run Rate Per Team Per Season
# ---------------------------------------------------
balls = df.groupby(['season','batting_team','match_id']) \
    .size() \
    .reset_index(name='Balls')

runs = df.groupby(['season','batting_team','match_id'])['runs_total'] \
    .sum() \
    .reset_index()

rr = pd.merge(runs, balls, on=['season','batting_team','match_id'])
rr['Run Rate'] = (rr['runs_total'] / rr['Balls']) * 6

avg_rr = rr.groupby(['season','batting_team'])['Run Rate'] \
    .mean() \
    .reset_index()

# ---------------------------------------------------
# 5️⃣ Total Sixes
# ---------------------------------------------------
sixes = df[df['runs_batter'] == 6] \
    .groupby(['season','batting_team']) \
    .size() \
    .reset_index(name='Total Sixes')

# ---------------------------------------------------
# 6️⃣ Total Fours
# ---------------------------------------------------
fours = df[df['runs_batter'] == 4] \
    .groupby(['season','batting_team']) \
    .size() \
    .reset_index(name='Total Fours')

# ---------------------------------------------------
# 7️⃣ Wickets Taken Per Season
# ---------------------------------------------------
wickets = df.groupby(['season','bowling_team'])['is_wicket'] \
    .sum() \
    .reset_index() \
    .rename(columns={'bowling_team':'batting_team',
                     'is_wicket':'Wickets Taken'})

# ---------------------------------------------------
# 8️⃣ Merge Everything
# ---------------------------------------------------
team_season_summary = matches \
    .merge(team_runs, on=['season','batting_team']) \
    .merge(avg_score, on=['season','batting_team']) \
    .merge(avg_rr, on=['season','batting_team']) \
    .merge(sixes, on=['season','batting_team'], how='left') \
    .merge(fours, on=['season','batting_team'], how='left') \
    .merge(wickets, on=['season','batting_team'], how='left')

# Replace NaN with 0
team_season_summary.fillna(0, inplace=True)

# Rename cleanly
team_season_summary.rename(columns={
    'batting_team':'Team'
}, inplace=True)

# Sort season-wise
team_season_summary = team_season_summary \
    .sort_values(['season','Team']) \
    .reset_index(drop=True)

team_season_summary

,season,Team,Matches Played,Total Runs,Avg Match Score,Run Rate,Total Sixes,Total Fours,Wickets Taken
0,2009.0,Chennai Super Kings,14,2231,159.357143,7.839805,73,192,91
1,2009.0,Deccan Chargers,16,2408,150.500000,7.638744,99,173,110
2,2009.0,Delhi Daredevils,15,2131,142.066667,7.645785,51,188,106
3,2009.0,Kings XI Punjab,14,1928,137.714286,6.925560,61,140,76
4,2009.0,Kolkata Knight Riders,13,1772,136.307692,6.997382,56,143,59
...,...,...,...,...,...,...,...,...,...
127,2025.0,Mumbai Indians,16,2912,182.000000,9.268215,143,260,116
128,2025.0,Punjab Kings,18,3262,181.222222,9.576896,179,267,97
129,2025.0,Rajasthan Royals,14,2614,186.714286,9.361502,146,211,68
130,2025.0,Royal Challengers Bengaluru,15,2653,176.866667,9.304451,125,239,94


In [41]:
df['is_dot'] = (df['runs_total'] == 0).astype(int)

dot_balls = df.groupby('season')['is_dot'].sum().reset_index(name='Dot Balls')
total_balls = df.groupby('season').size().reset_index(name='Total Balls')

dot_trend = dot_balls.merge(total_balls, on='season')
dot_trend['Dot Ball %'] = (
    (dot_trend['Dot Balls'] / dot_trend['Total Balls']) * 100
)

dot_trend.sort_values('season')
#dot ball trend

,season,Dot Balls,Total Balls,Dot Ball %
0,2009.0,5119,13606,37.623107
1,2011.0,6282,17013,36.924705
2,2012.0,6241,17767,35.126921
3,2013.0,6759,18177,37.184354
4,2014.0,5017,14300,35.083916
5,2015.0,4775,13652,34.976560
6,2016.0,4649,14096,32.980988
7,2017.0,4551,13862,32.830760
8,2018.0,4753,14286,33.270335
9,2019.0,4936,14312,34.488541


In [42]:
total_runs = df.groupby('season')['runs_total'].sum().reset_index(name='Total Runs')

boundary_runs = df[df['runs_batter'].isin([4,6])]
boundary_runs = (
    boundary_runs.groupby('season')['runs_batter']
    .sum()
    .reset_index(name='Boundary Runs')
)

evolution = total_runs.merge(boundary_runs, on='season')
evolution['Boundary %'] = (
    (evolution['Boundary Runs'] / evolution['Total Runs']) * 100
)

evolution.sort_values('season')
#boundary trend

,season,Total Runs,Boundary Runs,Boundary %
0,2009.0,16353,8332,50.950896
1,2011.0,21154,11498,54.353787
2,2012.0,22453,12042,53.632031
3,2013.0,22602,12302,54.428812
4,2014.0,18931,10542,55.686440
5,2015.0,18353,10596,57.734430
6,2016.0,18862,10366,54.957057
7,2017.0,18786,10684,56.872139
8,2018.0,19901,11840,59.494498
9,2019.0,19434,11336,58.330761


In [43]:
# Total runs per season
runs_trend = (
    df.groupby('season')['runs_total']
    .sum()
    .reset_index(name='Total Runs')
    .sort_values('season')
)

runs_trend

,season,Total Runs
0,2009.0,16353
1,2011.0,21154
2,2012.0,22453
3,2013.0,22602
4,2014.0,18931
5,2015.0,18353
6,2016.0,18862
7,2017.0,18786
8,2018.0,19901
9,2019.0,19434


In [44]:
balls = df.groupby(['season','match_id']).size().reset_index(name='Balls')
runs = df.groupby(['season','match_id'])['runs_total'].sum().reset_index()

rr = pd.merge(runs, balls, on=['season','match_id'])
rr['Run Rate'] = (rr['runs_total'] / rr['Balls']) * 6

runrate_trend = (
    rr.groupby('season')['Run Rate']
    .mean()
    .reset_index()
    .sort_values('season')
)

runrate_trend

,season,Run Rate
0,2009.0,7.227063
1,2011.0,7.411735
2,2012.0,7.573468
3,2013.0,7.470888
4,2014.0,7.922345
5,2015.0,8.134891
6,2016.0,8.000792
7,2017.0,8.088623
8,2018.0,8.362224
9,2019.0,8.177726


In [45]:
sixes = df[df['runs_batter'] == 6]

sixes_per_match = (
    sixes.groupby(['season','match_id'])
    .size()
    .reset_index(name='Sixes')
)

sixes_trend = (
    sixes_per_match.groupby('season')['Sixes']
    .mean()
    .reset_index(name='Avg Sixes Per Match')
    .sort_values('season')
)

sixes_trend

,season,Avg Sixes Per Match
0,2009.0,8.912281
1,2011.0,8.753425
2,2012.0,9.905405
3,2013.0,8.960526
4,2014.0,11.916667
5,2015.0,11.728814
6,2016.0,10.650000
7,2017.0,11.966102
8,2018.0,14.533333
9,2019.0,13.100000


In [47]:
scored = df.groupby(['season','batting_team'])['runs_total'].sum().reset_index()
conceded = df.groupby(['season','bowling_team'])['runs_total'].sum().reset_index()

conceded.rename(columns={'bowling_team':'batting_team',
                         'runs_total':'Runs Conceded'}, inplace=True)

strength = scored.merge(conceded, on=['season','batting_team'])
strength.rename(columns={'runs_total':'Runs Scored'}, inplace=True)

strength['Dominance Index'] = (
    (strength['Runs Scored'] - strength['Runs Conceded'])
)

strength.sort_values(['season','Dominance Index'], ascending=[True, False])

,season,batting_team,Runs Scored,Runs Conceded,Dominance Index
0,2009.0,Chennai Super Kings,2231,2004,227
5,2009.0,Mumbai Indians,1897,1802,95
3,2009.0,Kings XI Punjab,1928,1886,42
1,2009.0,Deccan Chargers,2408,2387,21
2,2009.0,Delhi Daredevils,2131,2158,-27
...,...,...,...,...,...
123,2025.0,Delhi Capitals,2500,2542,-42
126,2025.0,Lucknow Super Giants,2732,2779,-47
131,2025.0,Sunrisers Hyderabad,2519,2584,-65
129,2025.0,Rajasthan Royals,2614,2786,-172
